# 제주 설문 → 선호·필수 제약·표시 context

표준 라이브러리만 사용하는 **교육용 작은 변환기**입니다. 전체 앱·전체 설문·완전한 제약/통화 검증기가 아닙니다. 외부 API·추론·다운로드·개인정보 저장은 없습니다. 아래 응답과 USD 1단위당 1,400원 환율은 **허구·설명용 가정이며 실시간 시세가 아닙니다.**

지원 입력: Q07 최대 3개 테마 순위, Q06 활동 선호, PREF-C01 혼잡 회피, PREF-N01 새로움, PREF-I01 실내 선호, 1인 하루 원화 예산, 꼭 포함할 경험. B01 연령대·B02 성별·B03 국적·B04 표시 통화는 `context`에만 보존하며 **13개 선호 축·매칭 점수에 사용하지 않습니다.**

가입정보를 prefill할 수 있지만 사용자가 확인·수정해야 합니다. 미확인 prefill은 `confirmed=false`로 보존하고 후속 확인을 남깁니다. 국적은 복수·미응답이 가능하며 국적에서 통화를 강제 결정하지 않습니다. 표시 통화는 별도로 선택·수정합니다. 미응답·미확인 선택은 응답 원값을 바꾸지 않고 별도 표시용 통화에 KRW 기본값을 적용합니다. 이 예제의 통화 코드는 KRW/USD/EUR/JPY/CNY/GBP만 지원합니다.

`answered|unknown|unanswered|indifferent` 상태를 구분합니다. 미정·미응답·상관없음에 0 또는 0.5를 넣지 않습니다. 명시적 중간 선호만 0.5이고 Q07 미선택은 싫다는 뜻이 아닙니다. 날짜·교통·식이·실제 접근성·명시적 부정 선호·전체 일정 합산·충돌 검사는 구현하지 않았습니다.


In [1]:
import math
import json
from copy import deepcopy
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from datetime import date

THEMES = ("coast", "nature", "local_food", "cafe", "culture", "art", "outdoor", "learning", "neighborhood")
SCALARS = ("pace", "crowd_avoidance", "novelty", "indoor_preference")
STATUSES = {"answered", "unknown", "unanswered", "indifferent"}
WEIGHTS = {"themes": .60, "pace": .15, "crowd_avoidance": .10, "novelty": .10, "indoor_preference": .05}
QUESTIONS = {"themes": "Q07", "pace": "Q06", "crowd_avoidance": "PREF-C01", "novelty": "PREF-N01", "indoor_preference": "PREF-I01", "budget": "Q05", "experience": "F09"}
CONTEXT = {"B01": "age_group", "B02": "gender", "B03": "nationalities", "B04": "display_currency"}
CURRENCIES = {"KRW", "USD", "EUR", "JPY", "CNY", "GBP"}

def read_answer(raw):
    if raw is None:
        return {"status": "unanswered"}
    if not isinstance(raw, dict) or raw.get("status") not in STATUSES:
        raise ValueError("응답 status가 필요합니다")
    if raw["status"] != "answered" and raw.get("value") is not None:
        raise ValueError("미정·미응답·상관없음에 값을 부여하지 마세요")
    return deepcopy(raw)

def finite_number(value):
    return type(value) in (int, float) and math.isfinite(value)

def survey_to_profile(answers, *, survey_version="survey-draft-v2", profile_version=1):
    """순수 변환. 인구통계·표시 통화로 선호·점수를 추정하지 않는다."""
    if not isinstance(answers, dict) or set(answers) - (set(QUESTIONS) | set(CONTEXT) | {"legacy_walk"}):
        raise ValueError("지원하지 않는 입력 키가 있습니다")
    if not isinstance(survey_version, str) or not survey_version.strip() or type(profile_version) is not int or profile_version < 1:
        raise ValueError("유효한 설문/프로필 버전이 필요합니다")
    if answers.get("legacy_walk") not in (None, "none"):
        raise ValueError("이 예제는 legacy walk:none만 지원합니다")
    out = {"schemaVersion": "educational-profile-v2", "surveyVersion": survey_version,
           "profileVersion": profile_version, "preferences": {"themes": None}, "context": {},
           "responseStates": {}, "evidence": {}, "hardConstraints": {}, "softRequirements": {},
           "unanswered": [], "followups": []}
    for qid, field in CONTEXT.items():
        a = read_answer(answers.get(qid))
        source = a.get("source", "user_survey")
        confirmed = a.get("confirmed", False)
        if source not in ("user_survey", "signup_prefill") or type(confirmed) is not bool:
            raise ValueError("context 출처·확인 상태가 잘못되었습니다")
        value = a.get("value") if a["status"] == "answered" else None
        if a["status"] == "answered":
            if qid == "B03":
                if not isinstance(value, list) or not 1 <= len(value) <= 5 or any(not isinstance(x, str) or not x.strip() or len(x) > 80 for x in value) or len(set(value)) != len(value):
                    raise ValueError("국적은 중복 없는 문자열 목록입니다")
            elif not isinstance(value, str) or not value.strip() or len(value) > 40:
                raise ValueError("context 문자열을 확인하세요")
            if qid == "B04" and value not in CURRENCIES:
                raise ValueError("교육 예제의 지원 통화가 아닙니다")
            if not confirmed:
                out["followups"].append({"field": qid, "reason": "가입/입력 정보를 확인하거나 수정해 주세요"})
        out["context"][field] = {"status": a["status"], "value": value, "source": source, "confirmed": confirmed}
        out["evidence"][qid] = {"questionId": qid, "source": source, "confirmed": confirmed, "surveyVersion": survey_version}
    for key in QUESTIONS:
        a = read_answer(answers.get(key))
        out["responseStates"][key] = a["status"]
        out["evidence"][key] = {"questionId": QUESTIONS[key], "source": "explicit_user_survey", "status": a["status"], "surveyVersion": survey_version}
        if a["status"] == "unanswered":
            out["unanswered"].append(key)
        if key == "themes":
            if a["status"] == "answered":
                ranked = a.get("value")
                if not isinstance(ranked, list) or not 1 <= len(ranked) <= 3 or any(not isinstance(x, str) or x not in THEMES for x in ranked) or len(set(ranked)) != len(ranked):
                    raise ValueError("테마는 중복 없이 1~3개입니다")
                points = dict(zip(ranked, (3, 2, 1)))
                out["preferences"][key] = [points.get(t, 0) / sum(points.values()) for t in THEMES]
                out["evidence"][key]["rankedSelections"] = ranked
        elif key in SCALARS:
            value = a.get("value") if a["status"] == "answered" else None
            if a["status"] == "answered" and (not finite_number(value) or not 0 <= value <= 1):
                raise ValueError("scalar는 명시적 0~1 수치여야 합니다")
            out["preferences"][key] = value
        elif a["status"] == "answered":
            if type(a.get("strict")) is not bool or type(a.get("confirmed")) is not bool:
                raise ValueError("조건에는 strict/confirmed를 명시하세요")
            value = a.get("value")
            if key == "budget":
                if (a.get("unit"), a.get("scope")) != ("KRW", "per_person_per_day") or type(value) is not int or value < 0:
                    raise ValueError("예산은 1인 1일 KRW 기준 0 이상 정수입니다")
            elif value not in ("cafe_dessert", "olle", "hallasan"):
                raise ValueError("지원 포함 경험이 아닙니다")
            requirement = {k: a[k] for k in ("value", "unit", "scope", "strict", "confirmed") if k in a}
            if a["strict"] and not a["confirmed"]:
                out["followups"].append({"field": key, "reason": "필수 조건 사용자 확인 대기", "pending": requirement})
            else:
                out["hardConstraints" if a["strict"] else "softRequirements"][key] = requirement
    if answers.get("legacy_walk") == "none":
        out["legacyPreferences"] = {"walking": "low", "source": "legacy_walk_none"}
        out["followups"].append({"field": "activity_preference", "reason": "기존 '거의 안 걸어요'의 현재 활동 선호를 직접 확인해 주세요. 숫자 분/km 상한을 추정하지 않습니다."})
    out["evidenceCoverage"] = round(sum(w for k, w in WEIGHTS.items() if out["preferences"].get(k) is not None), 10)
    return out

def resolve_display_currency(context):
    """응답은 그대로 보존하고 표시용 통화만 결정한다. 미확인 선택은 원화 기본값."""
    choice = context.get("display_currency", {})
    if choice.get("status") == "answered" and choice.get("confirmed") is True:
        return {"currency": choice.get("value"), "reason": "confirmed_user_selection"}
    return {"currency": "KRW", "reason": "default_krw_without_confirmed_selection"}

def budget_display(krw, currency, *, rate_krw_per_unit=None, source=None, asof=None, assumed=False):
    """표시만 환산한다. 예산 판정의 원화 값은 변경하지 않는다."""
    if type(krw) is not int or krw < 0 or type(assumed) is not bool:
        raise ValueError("원화 금액·가정 여부가 잘못되었습니다")
    base = {"krw": krw, "currency": currency}
    if currency == "KRW":
        return {**base, "status": "available", "amount": str(krw), "notice": "원화 원값"}
    if currency not in CURRENCIES or rate_krw_per_unit is None or not source or not asof:
        return {**base, "status": "unavailable", "amount": None, "notice": "선택 통화 또는 환율·출처·기준일 확인 필요; 원화 유지"}
    if not isinstance(rate_krw_per_unit, (str, Decimal)) or not isinstance(source, str) or not isinstance(asof, str):
        raise ValueError("환율은 Decimal 또는 문자열, 출처·기준일은 문자열로 전달하세요")
    try:
        rate = Decimal(rate_krw_per_unit)
        date.fromisoformat(asof)
    except (InvalidOperation, ValueError):
        raise ValueError("환율·기준일 형식이 잘못되었습니다") from None
    if not rate.is_finite() or rate <= 0:
        raise ValueError("환율은 유한한 양수여야 합니다")
    quantum = Decimal("1") if currency == "JPY" else Decimal("0.01")
    amount = (Decimal(krw) / rate).quantize(quantum, rounding=ROUND_HALF_UP)
    return {**base, "status": "available", "amount": str(amount), "rateKRWPerUnit": str(rate),
            "source": source, "asOf": asof, "assumed": assumed,
            "notice": "설명용 가정 환율, 실시간 아님" if assumed else "제공된 환율의 기준일 표시; 실시간 여부는 별도 검증"}


테마는 순위 3·2·1을 선택 점수 합으로 나눕니다. 테마 9축 전체가 가중치 0.60인 한 블록이고 네 scalar는 0.15/0.10/0.10/0.05입니다. `evidenceCoverage`는 관측 블록의 정보 충족률이며 추천 확률이 아닙니다. 인구통계와 통화는 coverage에도 영향을 주지 않습니다.

확인된 `strict=true`만 필수 조건입니다. 확인 대기는 `followups`에 보존하고 해결 전 완성 추천에 사용하지 않습니다. 필수 카페 조건은 실제 후보·일정에서 후속 검증해야 합니다. 예산은 항공·숙박 제외, 식사·입장료·현지 이동비의 1인 1일 원화 기준이며 환산 금액은 표시용입니다. 공동 비용 배분·카드 수수료·실제 결제 환율은 지원하지 않습니다.


In [2]:
# 허구 응답과 가정 환율입니다. 실제 사용자·시세 데이터가 아닙니다.
example = {
    "B01": {"status": "answered", "value": "30대", "source": "signup_prefill", "confirmed": False},
    "B02": {"status": "unanswered"},
    "B03": {"status": "answered", "value": ["대한민국", "캐나다"], "source": "user_survey", "confirmed": True},
    "B04": {"status": "answered", "value": "USD", "source": "user_survey", "confirmed": True},
    "themes": {"status": "answered", "value": ["coast", "cafe", "culture"]},
    "pace": {"status": "answered", "value": 0},
    "crowd_avoidance": {"status": "answered", "value": .75},
    "novelty": {"status": "unknown"},
    "indoor_preference": {"status": "indifferent"},
    "budget": {"status": "answered", "value": 50000, "unit": "KRW", "scope": "per_person_per_day", "strict": False, "confirmed": True},
    "experience": {"status": "answered", "value": "cafe_dessert", "strict": True, "confirmed": True},
    "legacy_walk": "none"
}
profile = survey_to_profile(example)
resolved_currency = resolve_display_currency(profile["context"])
selected_currency = resolved_currency["currency"]
display = budget_display(50000, selected_currency, rate_krw_per_unit="1400", source="교육용 가정: 1 USD = 1,400 KRW", asof="2026-11-18", assumed=True)
print(json.dumps({"profile": profile, "resolvedDisplayCurrency": resolved_currency, "budgetDisplay": display}, ensure_ascii=False, indent=2))


{
  "profile": {
    "schemaVersion": "educational-profile-v2",
    "surveyVersion": "survey-draft-v2",
    "profileVersion": 1,
    "preferences": {
      "themes": [
        0.5,
        0.0,
        0.0,
        0.3333333333333333,
        0.16666666666666666,
        0.0,
        0.0,
        0.0,
        0.0
      ],
      "pace": 0,
      "crowd_avoidance": 0.75,
      "novelty": null,
      "indoor_preference": null
    },
    "context": {
      "age_group": {
        "status": "answered",
        "value": "30대",
        "source": "signup_prefill",
        "confirmed": false
      },
      "gender": {
        "status": "unanswered",
        "value": null,
        "source": "user_survey",
        "confirmed": false
      },
      "nationalities": {
        "status": "answered",
        "value": [
          "대한민국",
          "캐나다"
        ],
        "source": "user_survey",
        "confirmed": true
      },
      "display_currency": {
        "status": "answered",
        "value"

## 핵심 검사 6개 + 환율 표시 검사 2개
지원 범위만 검증합니다. 인구통계를 바꾸어도 선호와 coverage는 동일해야 합니다. 기존 '거의 안 걸어요'는 현재 활동 선호 확인으로 이어지며 분·km 상한을 만들지 않습니다.


In [3]:
def must_reject(fn, *args, **kwargs):
    try:
        fn(*args, **kwargs)
    except ValueError:
        return
    raise AssertionError("거절해야 할 입력입니다")

# 1. 순위 정규화·입력 불변성
before = deepcopy(example); p = survey_to_profile(example)
assert example == before and math.isclose(sum(p["preferences"]["themes"]), 1)
assert p["preferences"]["themes"][0] == .5
print("1 PASS: 3·2·1 정규화와 입력 보존")

# 2. 네 응답 상태와 명시적 0
assert p["preferences"]["pace"] == 0 and p["responseStates"]["novelty"] == "unknown"
assert p["responseStates"]["indoor_preference"] == "indifferent"
assert p["context"]["gender"]["status"] == "unanswered"
print("2 PASS: 미정·미응답·상관없음·명시적 0 구분")

# 3. 잘못된 선호 입력 거절
must_reject(survey_to_profile, {"themes": {"status": "answered", "value": ["coast", "coast"]}})
must_reject(survey_to_profile, {"pace": {"status": "answered", "value": True}})
must_reject(survey_to_profile, {"pace": {"status": "answered", "value": float("nan")}})
print("3 PASS: 중복·bool·비유한 수 거절")

# 4. 카페 필수·예산 선호·단위/범위
assert "experience" in p["hardConstraints"] and "budget" in p["softRequirements"]
bad = deepcopy(example); bad["budget"]["scope"] = "whole_trip"
must_reject(survey_to_profile, bad)
print("4 PASS: 필수 포함과 예산 선호 분리, 잘못된 범위 거절")

# 5. 필수 확인 대기·legacy 활동 선호 확인
pending = deepcopy(example); pending["experience"]["confirmed"] = False
r = survey_to_profile(pending)
assert "experience" not in r["hardConstraints"]
assert {f["field"] for f in r["followups"]} >= {"experience", "activity_preference", "B01"}
assert r["legacyPreferences"]["walking"] == "low" and "walking" not in r["hardConstraints"]
print("5 PASS: 미확인 필수/가입정보 후속 확인, 보행 수치 추정 없음")

# 6. context 독립·복수 국적·버전/근거
other = deepcopy(example)
other["B01"] = {"status": "answered", "value": "60대", "confirmed": True}
other["B02"] = {"status": "answered", "value": "응답 직접 입력", "confirmed": True}
other["B03"] = {"status": "unanswered"}
other["B04"] = {"status": "answered", "value": "JPY", "confirmed": True}
o = survey_to_profile(other)
assert p["preferences"] == o["preferences"] and p["evidenceCoverage"] == o["evidenceCoverage"] == .85
assert len(p["context"]["nationalities"]["value"]) == 2
assert o["context"]["display_currency"]["value"] == "JPY"
assert p["evidence"]["themes"]["questionId"] == "Q07" and p["surveyVersion"] == "survey-draft-v2"
assert survey_to_profile({})["evidenceCoverage"] == 0
print("6 PASS: 인구통계/통화는 13축·coverage와 독립, 버전·근거 보존")

# 7. 환율 방향·Decimal 반올림·가정/기준일 표시
f = budget_display(50000, "USD", rate_krw_per_unit="1400", source="교육용 가정", asof="2026-11-18", assumed=True)
assert f["amount"] == "35.71" and f["krw"] == 50000 and f["assumed"] is True
assert f["asOf"] == "2026-11-18" and "실시간 아님" in f["notice"]
print("7 PASS: KRW/외화 1단위 환율로 나누기, 가정 시세 표시")

# 8. 환율 미확인/무효 처리
assert budget_display(50000, "USD")["status"] == "unavailable"
assert budget_display(50000, None)["amount"] is None  # formatter 직접 호출의 미지원 값
missing = survey_to_profile({})
original_context = deepcopy(missing["context"])
resolved = resolve_display_currency(missing["context"])
assert resolved["currency"] == "KRW"
assert budget_display(50000, resolved["currency"])["amount"] == "50000"
assert missing["context"] == original_context
assert missing["context"]["display_currency"]["status"] == "unanswered"
assert missing["context"]["display_currency"]["value"] is None
unsupported = {"display_currency": {"status": "answered", "value": "ZZZ", "confirmed": True}}
assert budget_display(50000, resolve_display_currency(unsupported)["currency"])["status"] == "unavailable"
must_reject(budget_display, 50000, "USD", rate_krw_per_unit="0", source="예시", asof="2026-11-18")
assert budget_display(50000, "KRW")["amount"] == "50000"
print("8 PASS: 미응답은 별도 KRW 기본표시, 응답 보존; 환율 누락/미지원은 unavailable")


1 PASS: 3·2·1 정규화와 입력 보존
2 PASS: 미정·미응답·상관없음·명시적 0 구분
3 PASS: 중복·bool·비유한 수 거절
4 PASS: 필수 포함과 예산 선호 분리, 잘못된 범위 거절
5 PASS: 미확인 필수/가입정보 후속 확인, 보행 수치 추정 없음
6 PASS: 인구통계/통화는 13축·coverage와 독립, 버전·근거 보존
7 PASS: KRW/외화 1단위 환율로 나누기, 가정 시세 표시
8 PASS: 미응답은 별도 KRW 기본표시, 응답 보존; 환율 누락/미지원은 unavailable


## 앱 연결 전
입력 검증 → 순수 변환 → 가입정보·필수 조건·활동 선호 확인 → 페르소나 매칭 → 실제 후보 제약 검사 순으로 분리합니다. 기존 `scripts/persona_model.py`에는 `preferences`만 전달하며 `context`를 점수 입력으로 합치지 않습니다. 인구통계는 표시·가입정보 확인 목적에 필요한 범위에서 선택적으로 보존하고 사용 목적·보존 정책을 별도로 정합니다.

예산 판정 원값은 KRW로 유지하고 표시 통화 선택·수정만 반영합니다. B04 미응답은 context에 unanswered/null로 남기고 resolvedDisplayCurrency만 KRW로 결정합니다. 확인된 명시적 미지원 통화는 원화로 몰래 바꾸지 않고 unavailable로 처리합니다. 실제 환율 연동 시 출처·기준일·갱신 실패·수수료/시세 차이를 별도 처리해야 합니다. 이 함수의 `asOf`는 입력받은 날짜이며 최신 시세를 검증했다는 뜻이 아닙니다.

관련 문서: `docs/04-questionnaire-and-scoring.md`, `docs/08-persona-clustering.md`, `data/persona-prototypes.json`. 노트북은 독립 실행되며 외부 파일이 필요하지 않습니다.
